# 01. 量化基础知识

本章是 AMD Quark 官方 “Quantization with AMD Quark” 页面的中文翻译版，作为后续 Quark LLM 量化实战的概念铺垫。

> 说明：在官方文档中，**AMD Quark** 有时会简称为 **Quark**。除非另有说明，文中的 “Quark” 特指 AMD Quark 量化器，不要和其他同名产品或技术混淆。

## 什么是量化？

量化指的是：将一个数值从较高 bit 宽度的表示方式，压缩到较低 bit 宽度的表示方式，从而优化内存使用和计算效率。

为了说明这个概念，可以考虑数字 `1024`。

如果它被表示成一个普通整数，通常会占用 `32 bit`，也就是 `4 bytes`。在这种表示里，有超过 `2 bytes` 的内存，也就是从 bit 31 到 bit 11，会被用来存储 `0`。如果把这个数字转换成 `16 bit` 整数类型，我们仍然可以无损地表示同一个值，同时内存占用几乎减半。

这就是量化的一个简单例子。应用到神经网络时，量化可以让模型更小，使模型能够部署在成本更低的硬件上，并且有可能比原始全精度模型运行得更快。

不过，量化并不总是这么直接。

数值通常会以小数形式存在，并且经常遵循 IEEE-754 浮点数标准。由于这种表示方式比较复杂，我们不能简单地从这些数据类型里删掉几个字节就完成压缩。

例如，如果把 `1024` 表示为 `32 bit` 浮点数，就不再那么容易看出到底哪些 bit 可以被移除，且不会影响有效表示。

做量化时，有几个关键问题需要考虑：

- **量化可能是有损压缩**：精度可能会不可逆地丢失。例如，像 `1024.23` 这样的值，在较低 bit 宽度下可能必须近似成 `1024.0` 或 `1024.5`。
- **量化值参与算术运算时可能溢出**：这时可能需要更大的 bit 宽度来容纳结果。例如，`255` 可以用 `8 bit` 表示，即 `1111 1111`；但 `255 * 2 = 510` 需要 `9 bit`，即 `1 1111 1110`。
- **可用于量化的数据类型很多**：从 `16 bit` 到 `4 bit` 的整数，从 `16 bit` 到 `4 bit` 的浮点数，以及更高级的组合类型，例如 MX。

正因为这些细节比较复杂，使用 **AMD Quark** 这样的库会更有帮助，因为它可以替你管理这些底层复杂性。

## 模型量化

对于深度学习模型，量化通常指的是把模型的 **权重** 和 **激活值** 从浮点表示转换为较低 bit 宽度的浮点或整数表示。

这种转换可以显著降低模型的内存占用和计算需求，从而提升它在支持低 bit 数据类型的硬件上的运行效率。量化可能会影响模型精度，因此成功量化的关键，是在 **精度** 和 **性能** 之间取得平衡。

模型量化技术大致可以分为两类：

- **量化感知训练，Quantization-Aware Training，QAT**：在训练过程中引入量化，以尽量减少量化带来的精度损失。
- **训练后量化，Post-Training Quantization，PTQ**：在模型训练完成后应用量化，不需要重新训练。

PTQ 是量化预训练模型时很常用的方法，因为它不需要重新训练，并且可以应用到任意预训练模型。但它也可能因为量化过程导致模型精度下降。Quark 会通过多种训练后量化技术，帮助恢复量化后可能受损的模型精度。

从模型的量化表示来看，也可以大致分为两类：

- **量化到低 bit 浮点精度**：例如从 `float32` 转换到 `float16`、`bfloat16` 或 `float8`。这种情况下，数据仍然保持类似的浮点格式，只是 bit 宽度更低。当硬件支持这些低 bit 浮点数据类型时，这种方式会很有优势。
- **量化到整数精度**：例如从 `float32` 或 `float16` 转换到 `int8` 或 `int4`。这种情况下，数据会被映射到整数范围内。当硬件支持整数数据类型，并且能比浮点类型更高效地执行整数运算时，这种方式会更有收益。

## 量化时需要考虑什么？

思考量化方案时，需要注意以下因素：

- **目标硬件**：不同硬件，例如 CPU 和 GPU，对低精度计算的处理方式可能不同。需要选择适合目标硬件的量化方法。
- **模型特征**：模型的类型和复杂度会影响它承受量化的能力。更复杂的模型有时可能比简单模型更能适应量化。
- **使用场景要求**：不同应用对精度的要求不同。理解具体业务场景的精度要求，有助于选择合适的量化方法。

## 整数量化

将真实数值映射到整数，需要计算量化参数，通常称为 **scale** 和 **offset**。

量化公式：

```text
q = round(r / s + z)
```

反量化公式：

```text
r = (q - z) * s
```

在这些公式中：

- `q` 是量化后的值。
- `r` 是真实值。
- `s` 是 scale。
- `z` 是 offset。

为了计算这些量化参数，需要观察 tensor 的真实取值，并确定它的最小值和最大值。随后，scale 和 offset 会根据这些值计算出来。

Quark 提供了多种算法，帮助你调整这些量化参数，从而达到想要的精度和性能平衡。

量化可以根据 offset 的取值进行分类：

- **对称量化，symmetric quantization**：`z = 0`
- **非对称量化，asymmetric quantization**：`z != 0`

另外，根据量化参数，也就是 scale 和 offset，是如何为 tensor 中的元素计算的，量化还可以分为：

- **per-tensor**：整个 tensor 共用一组量化参数。
- **per-channel**：每个 channel 使用一组量化参数。
- **per-group**：每个 group 使用一组量化参数。

Quark 在不同框架中支持不同的量化 scheme。更多细节可以参考 Quark PyTorch / ONNX 用户指南。

## 伪量化，Fake Quantization

为了简化量化模型的操作，并支持那些硬件层面可能还不支持的数据类型，Quark 使用一种叫做 **模拟量化，simulated quantization** 的技术，也叫 **伪量化，fake quantization**。

当你在 Quark 中执行量化时，数值并不会直接被真正转换成低 bit 类型，而是会先经过 **fake quantization**。

## 这意味着什么？

这意味着数值不会立刻转换成新的量化数据类型。例如，一个 `32 bit` 权重不会马上以 `8 bit` 整数的形式存储在内存中。相反，它暂时仍然保持原始 bit 宽度。

量化过程会通过“先压缩，再解压回原始宽度”的方式进行模拟。最终结果仍然是原始 bit 宽度，但这个值本身已经可以用较低 bit 宽度的数据类型表示。

这种方式允许你在 Quark 中用较高 bit 宽度对量化模型执行推理，因此不需要硬件真正支持目标量化数据类型。不过，推理结果可以反映最终完成量化后大致可以期待的精度。当然，Quark也支持实际的低 bit 量化模型在vllm, sglang上推理。

## 量化时发生了什么？

当你把一个模型传给 Quark 进行量化时，最开始的步骤之一，是将某些层替换成 Quark 的等价层。

目前，Quark 为以下层提供了可替换的量化层：

- `Linear`
- `Conv2d`

当 Quark 遇到这些层类型时，会把它们替换为对应的 `Quant` 版本，例如 `QuantLinear` 或 `QuantConv2d`。

根据所选择的量化配置，这些新层可以用 fake quantized 版本拦截输入、输出、bias 和权重。

随后，在初始化 quantizer 时提供给 Quark 的校准数据，会被传入模型执行前向传播。

用户可定义的 observer 会在数据流经模型时接收这些数据，并计算正确量化所需的代表性最小值和最大值。

官方文档中提到的 observer 包括：

- `PerTensorMinMaxObserver`
- `PerChannelMinMaxObserver`
- `PerBlockMXObserver`

## 本章小结

这一章要带走的核心概念：

- 量化的本质是用更低 bit 宽度表示数值，以换取更低内存占用和更高计算效率。
- 量化可能带来精度损失，因此要在精度和性能之间做权衡。
- LLM 实战里常见的是 PTQ，因为它不需要重新训练模型。
- 整数量化离不开 `scale` 和 `zero point / offset`。
- Quark 中的 fake quantization 会先模拟低 bit 效果，不一定立即把权重物理存成低 bit。
- Quark 会替换部分层，例如 `Linear`、`Conv2d`，并通过 observer 和 calibration data 统计量化参数。

下一章我们进入高级量化方法与配置，把这些基础概念连接到 Quark 的实际配置系统。